In [ ]:
import logging
import time
import os
import sys

# --- 1. АВТОМАТИЧЕСКАЯ ПЕРЕЗАГРУЗКА МОДУЛЕЙ (%autoreload) ---
# Это позволяет вам изменять файлы в папках bots/, modules/ и config/ 
# без необходимости перезапускать ядро Jupyter.
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except Exception:
    pass # Пропускаем, если не в среде IPython/JN

# --- 2. ИМПОРТ КЛАССОВ ИЗ МОДУЛЕЙ (СКОРРЕКТИРОВАННЫЕ ПУТИ) ---
# Пути импорта скорректированы под структуру: modules/, bots/, config/
from modules.notifier import TelegramNotifier
from modules.decision_maker import DecisionMaker
from modules.mexc_clicker import MexcClicker # Предполагаем, что MexcClicker находится в bots/

# --- 3. НАСТРОЙКА ЛОГИРОВАНИЯ ---
def setup_logging(log_dir='log'):
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
        
    formatter = logging.Formatter(
        '%(asctime)s [%(name)s]: %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.INFO)
    
    # ОЧИСТКА: Удаляем существующие обработчики, чтобы избежать дублирования
    if root_logger.handlers:
        root_logger.handlers.clear()
        
    # Файловый обработчик
    log_file = os.path.join(log_dir, f"bot_log_{time.strftime('%Y%m%d_%H%M%S')}.log")
    file_handler = logging.FileHandler(log_file, encoding='utf-8')
    file_handler.setFormatter(formatter)
    
    # Консольный обработчик
    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    
    root_logger.addHandler(file_handler)
    root_logger.addHandler(stream_handler)
    
    return root_logger

LOGGER = setup_logging()

# --- 4. ГЛАВНЫЙ ЦИКЛ ЗАПУСКА ---

LOGGER.info("------ ЗАПУСК ОСНОВНОГО ЦИКЛА БОТА ------")

# 1. Инициализация сервисов
try:
    LOGGER.info("Инициализация Notifier, DecisionMaker и загрузка конфигов...")
    
    # Классы, наследующие BaseConfigLoader, загружают конфиги
    notifier = TelegramNotifier(LOGGER) 
    signal_gen = DecisionMaker(LOGGER, notifier)
    
except Exception as e:
    LOGGER.critical(f"❌ Критическая ошибка инициализации конфигурации/сервисов: {e}", exc_info=True)
    # В JN лучше вызвать исключение, чтобы остановить ячейку
    raise RuntimeError("Бот остановлен из-за ошибки инициализации.") from e

# 2. Создание и запуск кликера
try:
    bot = MexcClicker(LOGGER, notifier, signal_gen)
    LOGGER.info("✅ Инициализация успешна. Запуск основного цикла MexcClicker.")
    
    # Запуск основного цикла с интервалом 1 секунда и задержкой для переключения на вкладку
    time.sleep(15)
    bot.run_loop(interval_seconds=1)

except Exception as e:
    # Обработка непредвиденных ошибок, которые могут остановить run_loop
    error_message = (
        f"❌ <b>КРИТИЧЕСКАЯ ОШИБКА БОТА:</b> <code>{type(e).__name__}: {e}</code>. "
        f"Бот остановлен. Смотрите логи для полного Traceback."
    )
    LOGGER.critical(error_message, exc_info=True)
    
    # Отправляем уведомление с HTML форматированием
    notifier.notify(error_message)
    
    LOGGER.info("Бот завершил работу.")

2025-12-13 01:40:12 [root]: INFO - ------ ЗАПУСК ОСНОВНОГО ЦИКЛА БОТА ------
2025-12-13 01:40:12 [root]: INFO - Инициализация Notifier, DecisionMaker и загрузка конфигов...
2025-12-13 01:40:12 [root]: INFO - Загрузка конфигурационных файлов...
2025-12-13 01:40:12 [numexpr.utils]: INFO - NumExpr defaulting to 8 threads.
2025-12-13 01:40:12 [root]: INFO - Tesseract путь установлен.
2025-12-13 01:40:12 [Notifier]: INFO - Telegram Notifier инициализирован.
2025-12-13 01:40:12 [root]: INFO - Загрузка конфигурационных файлов...
2025-12-13 01:40:12 [root]: INFO - Tesseract путь установлен.
2025-12-13 01:40:12 [root]: INFO - Загрузка конфигурационных файлов...
2025-12-13 01:40:13 [root]: INFO - Tesseract путь установлен.
2025-12-13 01:40:13 [MexcClicker]: INFO - Контейнер свечей: 121 свечей (Live API).
2025-12-13 01:40:14 [root]: INFO - ✅ Инициализация успешна. Запуск основного цикла MexcClicker.
2025-12-13 01:40:29 [MexcClicker]: INFO - Начало основного цикла с интервалом 1 сек.
2025-12-13 01